## Quantizing hugging face pretrained model
https://huggingface.co/docs/peft/en/developer_guides/quantization \
https://huggingface.co/docs/transformers/quantization

In [ ]:
!pip install transformers torch bitsandbytes accelerate

In [ ]:
# LOAD TOKENIZER
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("tiiuae/falcon-7b")

###using 16 bit precision

In [ ]:
# using 16 bit precision
from transformers import AutoModelForCausalLM
import transformers, torch
model = AutoModelForCausalLM.from_pretrained("tiiuae/falcon-7b", torch_dtype=torch.bfloat16)

### using 8 bit precision

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import transformers, torch

config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained("tiiuae/falcon-7b", quantization_config=config)
gbs = model.get_memory_footprint() / 1e9
print(f"Number of parameters: {model.num_parameters()}")
print(f"Memory req if FP32: {(model.num_parameters()*4)/1e9} GB")
print(f"Memory req of the model with 8bit quantization: {gbs:.2f} GB")

In [ ]:
prompt = "Girafatron is obsessed with giraffes, the most glorious animal on the face of this Earth. Giraftron believes all other animals are irrelevant when compared to the glorious majesty of the giraffe.\nDaniel: Hello, Girafatron!\nGirafatron:"
prompt

In [ ]:
def generate(prompt):
  tokenized_text = tokenizer(prompt, return_tensors="pt").to("cuda")
  output = model.generate(**tokenized_text, eos_token_id=tokenizer.eos_token_id, do_sample=True, max_new_tokens=100)
  result = tokenizer.batch_decode(output,  skip_special_tokens=True)[0]
  return result

result = generate(prompt)
print(result)

###using 4 bit precision

In [ ]:
pip install -U bitsandbytes

In [ ]:
import torch
from transformers import BitsAndBytesConfig

config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

In [ ]:
#torch.cuda.empty_cache()
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers

four_bit_model = AutoModelForCausalLM.from_pretrained("tiiuae/falcon-7b", quantization_config=config)

In [ ]:
gbs = four_bit_model.get_memory_footprint() / 1e9

print(f"Number of parameters: {four_bit_model.num_parameters()}")
print(f"Memory footprint if FP32: {(four_bit_model.num_parameters()*4)/1e9} GB")
print(f"Memory footprint of the model with 4bit quantization: {gbs:.2f} GB")

In [ ]:
def generate(prompt):
  tokenized_text = tokenizer(prompt, return_tensors="pt").to("cuda")
  output = four_bit_model.generate(**tokenized_text, eos_token_id=tokenizer.eos_token_id, do_sample=True, max_new_tokens=100)
  result = tokenizer.batch_decode(output,  skip_special_tokens=True)[0]
  return result

result = generate(prompt)
print(result)